In [3]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   --------- ------------------------------ 1.8/8.0 MB 10.1 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.0 MB 12.4 MB/s eta 0:00:01
   ----------------------------------- ---- 7.1/8.0 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 12.1 MB/s  0:00:00
   ---------------------------------------- 0.0/38.6 MB ? eta -:--:--
   - -------------------------------------- 1.8/38.6 MB 11.2 MB/s eta 0:00:04
   ---- ----------------------------------- 4.7/38.6 MB 12.4 MB/s eta 0:00:03
   ------- -------------------------------- 7.6/38.6 MB 13.0 MB/s eta 0:00:03
   ---------- ----------------------------- 10.2/38.6 MB 12.7 MB/s eta 0:00:03
   ------------- -------------------------- 13.1/38.6 MB 13.0 MB/s eta 0:00:02
   --------------- ------------------------ 15.2/38.6 MB 12.6 MB/s eta 0:00:02
   ------------------ --------------------- 18.1/38.6 MB 12.7 MB/s eta 0:00:02
   ---


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ===============================================
# Centralized Split (Auto Path, IPYNB Friendly)
# ===============================================

import pandas as pd
from sklearn.model_selection import train_test_split
import os

# -------------------------
# AUTO PATH: Directory Notebook
# -------------------------
current_dir = os.getcwd()
print("Notebook directory:", current_dir)

# Path dataset tokenized (file harus ada di folder yang sama)
DATA_FILE = os.path.join(current_dir, "hatespeech_indobert_tokenized.pkl")

# Folder output split
OUT_DIR = os.path.join(current_dir, "centralized_split")
os.makedirs(OUT_DIR, exist_ok=True)

print("Output will be saved to:", OUT_DIR)

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_pickle(DATA_FILE)

label_cols = [
    "HS", "Abusive", "HS_Individual", "HS_Group", "HS_Religion",
    "HS_Race", "HS_Physical", "HS_Gender", "HS_Other",
    "HS_Weak", "HS_Moderate", "HS_Strong"
]

X_ids  = df["input_ids"].tolist()
X_mask = df["attention_mask"].tolist()
Y      = df[label_cols].values

# Stratify pakai label paling balance (Abusive)
stratify_label = df["Abusive"]

# -------------------------
# SPLIT 1 — Train 70%, Temp 30%
# -------------------------
(
    X_ids_train, X_ids_temp,
    X_mask_train, X_mask_temp,
    y_train, y_temp,
    strat_train, strat_temp
) = train_test_split(
    X_ids, X_mask, Y, stratify_label,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# -------------------------
# SPLIT 2 — Temp → Val 15%, Test 15%
# -------------------------
(
    X_ids_val, X_ids_test,
    X_mask_val, X_mask_test,
    y_val, y_test
) = train_test_split(
    X_ids_temp, X_mask_temp, y_temp,
    test_size=0.50,
    random_state=42,
    shuffle=True
)


# -------------------------
# FUNCTION TO SAVE SPLIT
# -------------------------
def save_split(name, ids, mask, labels):
    df_out = pd.DataFrame({
        "input_ids": ids,
        "attention_mask": mask
    })
    label_df = pd.DataFrame(labels, columns=label_cols)

    df_out = pd.concat([df_out, label_df], axis=1)

    save_path = os.path.join(OUT_DIR, f"{name}.pkl")
    df_out.to_pickle(save_path)

    print(f"[SAVED] {name}.pkl  → {df_out.shape}")


# -------------------------
# SAVE ALL 3 SPLITS
# -------------------------
save_split("train", X_ids_train, X_mask_train, y_train)
save_split("val",   X_ids_val,   X_mask_val,   y_val)
save_split("test",  X_ids_test,  X_mask_test,  y_test)

print("\n============== SPLIT COMPLETE ==============")
print("Train:", len(X_ids_train))
print("Val  :", len(X_ids_val))
print("Test :", len(X_ids_test))
print("============================================")


Notebook directory: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi
Output will be saved to: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi\centralized_split
[SAVED] train.pkl  → (9017, 14)
[SAVED] val.pkl  → (1932, 14)
[SAVED] test.pkl  → (1933, 14)

============== SPLIT COMPLETE ==============
Train: 9017
Val  : 1932
Test : 1933
